# Jaccard Similarity

Notebook ini mendemonstrasikan perhitungan **Jaccard similarity** antara query dan koleksi dokumen.

**Rumus Jaccard:**

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- Nilai berada pada rentang $[0, 1]$.
- $J = 1$ → himpunan term identik; $J = 0$ → tidak ada term yang sama.
- Mengabaikan frekuensi term (hanya keberadaan).

**Alur:**
1. Definisi koleksi dokumen & query.
2. Preprocessing (NLTK stopword + Sastrawi stemmer).
3. Hitung Jaccard query terhadap setiap dokumen.
4. Ranking dokumen.

## 1. Import Library & Inisialisasi

In [1]:
import re
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download('stopwords', quiet=True)
stop_id = set(stopwords.words('indonesian'))
stemmer = StemmerFactory().create_stemmer()
print('Jumlah stopword Indonesia (NLTK):', len(stop_id))

Jumlah stopword Indonesia (NLTK): 757


## 2. Koleksi Dokumen & Query

In [2]:
documents = {
    'D1': 'Mahasiswa belajar sistem temu kembali informasi.',
    'D2': 'Sistem informasi mengelola data mahasiswa di kampus.',
    'D3': 'Temu kembali dokumen menggunakan inverted index.',
    'D4': 'Mahasiswa mengakses dokumen melalui sistem temu kembali.',
}
query = 'sistem temu kembali mahasiswa'

print('Query:', query)
for doc_id, text in documents.items():
    print(f'{doc_id}: {text}')

Query: sistem temu kembali mahasiswa
D1: Mahasiswa belajar sistem temu kembali informasi.
D2: Sistem informasi mengelola data mahasiswa di kampus.
D3: Temu kembali dokumen menggunakan inverted index.
D4: Mahasiswa mengakses dokumen melalui sistem temu kembali.


## 3. Preprocessing

Lowercase → hapus tanda baca → tokenisasi → stopword removal → stemming. Hasilnya dikonversi menjadi **set** karena Jaccard berbasis himpunan.

In [3]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_id]
    tokens = [stemmer.stem(t) for t in tokens]
    return set(tokens)

doc_sets = {doc_id: preprocess(text) for doc_id, text in documents.items()}
query_set = preprocess(query)

print('Query  :', sorted(query_set))
for doc_id, s in doc_sets.items():
    print(f'{doc_id}     : {sorted(s)}')

Query  : ['mahasiswa', 'sistem', 'temu']
D1     : ['ajar', 'informasi', 'mahasiswa', 'sistem', 'temu']
D2     : ['data', 'informasi', 'kampus', 'kelola', 'mahasiswa', 'sistem']
D3     : ['dokumen', 'index', 'inverted', 'temu']
D4     : ['akses', 'dokumen', 'mahasiswa', 'sistem', 'temu']


## 4. Hitung Jaccard Similarity

In [4]:
def jaccard(a, b):
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)

scores = {}
print(f'{"Doc":<5}{"Irisan (A∩B)":<35}{"Gabungan (A∪B)":<45}{"Jaccard"}')
print('-' * 100)
for doc_id, doc_set in doc_sets.items():
    inter = query_set & doc_set
    union = query_set | doc_set
    score = jaccard(query_set, doc_set)
    scores[doc_id] = score
    print(f'{doc_id:<5}{str(sorted(inter)):<35}{str(sorted(union)):<45}{score:.4f}')

Doc  Irisan (A∩B)                       Gabungan (A∪B)                               Jaccard
----------------------------------------------------------------------------------------------------
D1   ['mahasiswa', 'sistem', 'temu']    ['ajar', 'informasi', 'mahasiswa', 'sistem', 'temu']0.6000
D2   ['mahasiswa', 'sistem']            ['data', 'informasi', 'kampus', 'kelola', 'mahasiswa', 'sistem', 'temu']0.2857
D3   ['temu']                           ['dokumen', 'index', 'inverted', 'mahasiswa', 'sistem', 'temu']0.1667
D4   ['mahasiswa', 'sistem', 'temu']    ['akses', 'dokumen', 'mahasiswa', 'sistem', 'temu']0.6000


## 5. Ranking Dokumen

Dokumen diurutkan berdasarkan skor Jaccard tertinggi.

In [5]:
ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
print('Peringkat:')
for rank, (doc_id, score) in enumerate(ranked, 1):
    print(f'  {rank}. {doc_id}  -> Jaccard = {score:.4f}')

Peringkat:
  1. D1  -> Jaccard = 0.6000
  2. D4  -> Jaccard = 0.6000
  3. D2  -> Jaccard = 0.2857
  4. D3  -> Jaccard = 0.1667


## 6. Ringkasan

- Jaccard similarity **murni berbasis himpunan**: hanya melihat term apa saja yang muncul, bukan berapa kali muncul.
- Cocok untuk perbandingan cepat (mis. *near-duplicate detection*), tetapi kurang sensitif terhadap kepentingan term.
- Untuk perankingan yang memperhitungkan frekuensi & kekhasan term, gunakan **TF-IDF** atau **VSM (cosine similarity)**.